In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Import libraries 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import math

from sklearn.preprocessing import StandardScaler, OneHotEncoder ,OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn import set_config
set_config(transform_output='pandas')

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')


# Load and inspect data

In [ ]:
df=pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

# EDA

In [ ]:


# Features and target
features = [
    "PitStop", "LapNumber", "Stint", "TyreLife", "Position",
    "LapTime (s)", "LapTime_Delta", "Cumulative_Degradation",
    "RaceProgress", "Position_Change"
]

target = "PitNextLap"

# Calculate grid size
n = len(features)
cols = 3  # number of columns in subplot grid
rows = math.ceil(n / cols)

# Create subplots
fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten()

# Plot each feature vs target
for i, feature in enumerate(features):
    axes[i].scatter(df[feature], df[target], alpha=0.5)
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel(target)
    axes[i].set_title(f"{feature} vs {target}")

# Remove extra empty plots (if any)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Categorical features and target
features = ["Driver", "Compound", "Race", "Year"]
target = "PitNextLap"

# Grid layout
n = len(features)
cols = 2
rows = math.ceil(n / cols)

# Create subplots
fig, axes = plt.subplots(rows, cols, figsize=(15, 6 * rows))
axes = axes.flatten()

# Plot boxplots
for i, feature in enumerate(features):
    sns.boxplot(x=df[feature], y=df[target], ax=axes[i])
    axes[i].set_title(f"{feature} vs {target}")
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel(target)
    axes[i].tick_params(axis='x', rotation=45)

# Remove extra plots if any
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


# Preprocessing

## Feature Engineering

In [ ]:
def engineer_features(df):
    import numpy as np
    
    df = df.copy()

    # ✅ 1. Sort for time consistency
    df = df.sort_values(["Driver", "Race", "Year", "LapNumber"])

    # ✅ 2. Race Progress
    max_lap = df.groupby(["Race", "Year"])["LapNumber"].transform("max")
    df["RaceProgress"] = df["LapNumber"] / max_lap

    # ✅ 3. Tire Degradation (stable)
    df["Deg_per_Lap"] = df["Cumulative_Degradation"] / (df["TyreLife"] + 1)
    df["Deg_Rolling"] = df.groupby(["Driver","Race","Year"])["Deg_per_Lap"]\
                           .transform(lambda x: x.rolling(5, min_periods=1).mean())

    # ✅ 4. Rolling Lap Time (IMPORTANT)
    df["LapTime_RollMean_3"] = df.groupby(["Driver","Race","Year"])["LapTime (s)"]\
                                .transform(lambda x: x.rolling(3, min_periods=1).mean())

    df["LapTime_RollMean_5"] = df.groupby(["Driver","Race","Year"])["LapTime (s)"]\
                                .transform(lambda x: x.rolling(5, min_periods=1).mean())

    df["LapTime_RollStd_5"] = df.groupby(["Driver","Race","Year"])["LapTime (s)"]\
                               .transform(lambda x: x.rolling(5, min_periods=2).std())

    # ✅ 5. Lag Features
    df["Lag1"] = df.groupby(["Driver","Race","Year"])["LapTime (s)"].shift(1)
    df["Lag3"] = df.groupby(["Driver","Race","Year"])["LapTime (s)"].shift(3)

    # ✅ 6. Lap Dynamics
    df["LapTime_Change"] = df.groupby(["Driver","Race","Year"])["LapTime (s)"].diff()
    df["Sudden_Drop"] = (df["LapTime_Change"] > 1.2).astype(int)

    # ✅ 7. Relative Pace
    race_avg = df.groupby(["Race","Year","LapNumber"])["LapTime (s)"].transform("mean")
    df["Rel_Pace"] = df["LapTime (s)"] - race_avg
    df["Rel_Pace_Scaled"] = df["Rel_Pace"] / (race_avg + 1e-5)

    # ✅ 8. Driver Consistency
    grp = df.groupby(["Driver","Race","Year"])["LapTime (s)"]
    df["Driver_Mean"] = grp.transform("mean")
    df["Driver_Std"] = grp.transform("std")
    df["Consistency_Index"] = df["Driver_Std"] / (df["Driver_Mean"] + 1e-5)

    # ✅ 9. Stint Features
    stint_len = df.groupby(["Driver","Race","Year","Stint"])["LapNumber"].transform("count")
    df["Stint_Length"] = stint_len
    df["Stint_Progress"] = df["TyreLife"] / (stint_len + 1)
    df["Tyre_Used_Ratio"] = df["TyreLife"] / (stint_len + 1)
    df["Is_New_Tyre"] = (df["TyreLife"] <= 2).astype(int)

    # ✅ 10. Pit Awareness
    df["Laps_Since_Pit"] = df.groupby(["Driver","Race","Year"])["PitStop"].cumsum()
    df["Just_Pitted"] = df.groupby(["Driver","Race","Year"])["PitStop"].shift(1).fillna(0)

    # ✅ 11. Position Features
    df["Position_Trend"] = df.groupby(["Driver","Race","Year"])["Position"].diff()
    df["Is_Gaining"] = (df["Position_Trend"] < 0).astype(int)
    df["Top3"] = (df["Position"] <= 3).astype(int)

    # ✅ 12. Compound-derived signals (NO encoding here!)
    # Only domain-based signals, no numeric mapping

    df["Is_Wet"] = df["Compound"].isin(["WET", "INTERMEDIATE"]).astype(int)
    df["Is_Slick"] = df["Compound"].isin(["SOFT", "MEDIUM", "HARD"]).astype(int)

    # simple grip tiers (optional but useful)
    df["High_Grip"] = (df["Compound"] == "SOFT").astype(int)
    df["Low_Grip"] = (df["Compound"].isin(["HARD", "WET"])).astype(int)

    # ✅ 13. Interaction Features (safe ones)
    df["TyreLife_x_Degradation"] = df["TyreLife"] * df["Cumulative_Degradation"]
    df["RelPace_x_TyreLife"] = df["Rel_Pace"] * df["TyreLife"]
    df["Position_x_RaceProgress"] = df["Position"] * df["RaceProgress"]

    # ✅ 14. Cleaning
    df = df.replace([np.inf, -np.inf], 0)
    df = df.fillna(0)

    return df

In [ ]:
df = engineer_features(df)

## define X and y

In [ ]:
# Create the input feature matrix
X = df.drop(["PitNextLap", "id"], axis=1)

# Create the target vector
y = df ["PitNextLap"]

## split data

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:


# Ordinal feature
ordinal_feature = ["Compound"]
compound_order = [["WET", "INTERMEDIATE", "SOFT", "MEDIUM", "HARD"]]

ordinal_encoder = OrdinalEncoder(categories=compound_order)

# Categorical features
cat_selector = ["Driver", "Race", "Year"]
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Numeric features
num_cols = X_train.select_dtypes("number").columns
scaler = StandardScaler()

# ----------- COLUMN TRANSFORMER -----------

preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal', ordinal_encoder, ordinal_feature),
        ('categorical', ohe, cat_selector),
        ('numeric', scaler, num_cols)
    ]
)



## COLUMN TRANSFORMER

In [ ]:
preprocessor.fit(X_train)
x_train_transformed=preprocessor.transform(X_train)
x_test_transformed=preprocessor.transform(X_test)

# Build LightGBM model

In [ ]:


model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=64,
    max_depth=10,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=2,
    random_state=42,
    class_weight='balanced'
)


model.fit(
    x_train_transformed, y_train,
    eval_metric='auc'
)



y_pred_proba = model.predict_proba(x_test_transformed)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)

print(f"AUC Score: {auc:.4f}")

## Test data

In [ ]:
df2=pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')
df2 = engineer_features(df2)
new_x = preprocessor.transform(df2)

In [ ]:
pred_probs = model.predict(new_x) 
pred = (pred_probs >= 0.5).astype(int)



tensorflow = pd.DataFrame({
    'id': df2['id'].values,
    'PitNextLap':pred.ravel()
})



tensorflow.to_csv('/kaggle/working/sample_submission.csv', index=False)


In [ ]:
df3=pd.read_csv('/kaggle/working/sample_submission.csv')
df3['PitNextLap'].value_counts()

# Tunning LightGBM model

In [ ]:
pip install optuna

In [ ]:
import optuna
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "max_depth": trial.suggest_int("max_depth", 4, 15),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "class_weight": "balanced"
    }

    model = LGBMClassifier(**params)


    model.fit(x_train_transformed, y_train)

    y_pred_proba = model.predict_proba(x_test_transformed)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)

    return auc


In [ ]:
import optuna

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)
print("Best AUC:", study.best_value)
print("Best parameters:", study.best_params)

In [ ]:
best_params = study.best_params

final_model = LGBMClassifier(
    **best_params,
    random_state=42,
    class_weight="balanced"
)

final_model.fit(x_train_transformed, y_train)


y_pred_proba = final_model.predict_proba(x_test_transformed)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Final Test AUC: {test_auc:.4f}")


## Test data

In [ ]:
red_probs = final_model.predict(new_x) 
pred = (pred_probs >= 0.5).astype(int)



tensorflow = pd.DataFrame({
    'id': df2['id'].values,
    'PitNextLap':pred.ravel()
})



tensorflow.to_csv('/kaggle/working/sample_submission2.csv', index=False)

In [ ]:
df4=pd.read_csv('/kaggle/working/sample_submission2.csv')
df4['PitNextLap'].value_counts()

# Neural Network

In [ ]:
print("x_train_transformed shape:", x_train_transformed.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid

In [ ]:
from tensorflow.keras import regularizers
model = tf.keras.Sequential([
        tf.keras.layers.Dense(
        128, activation='relu',
        input_shape=(942,),
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.Dropout(0.3), 
    tf.keras.layers.Dense(
        64, activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.Dropout(0.3),          
    tf.keras.layers.Dense(
        32, activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['AUC']
)

In [ ]:
history = model.fit(
    x_train_transformed, y_train,
    epochs=40,               
    validation_data=(x_test_transformed, y_test),
    batch_size=32
)

In [ ]:
df2=pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')
df2 = engineer_features(df2)
new_x = preprocessor.transform(df2)

pred_probs = model.predict(new_x) 
pred = (pred_probs >= 0.5).astype(int)
tensorflow = pd.DataFrame({
    'id': df2['id'].values,
    'PitNextLap':pred.ravel()
})

tensorflow.to_csv('/kaggle/working/sample_submission3.csv', index=False)
df3=pd.read_csv('/kaggle/working/sample_submission3.csv')
df3['PitNextLap'].value_counts()